### DAY 3 (22/02/26) – Job Orchestration Basics
####🏗️ Architecture & Strategy
Welcome to Day 3! Today, we transition from being "Interactive Data Scientists" to "Production ML Engineers."

In the real world, you do not sit at your computer and manually click "Run All" on notebooks every morning. Production systems require **Automation, Parameterization, and Orchestration.**

####Our Strategy:

**Parameterization** (`dbutils.widgets`): Hardcoding variables (like table names or pipeline stages) is a terrible practice. We will use Databricks Widgets to pass parameters into our notebook dynamically. This allows a single notebook to act as a universal controller.

**Modularization**: We will wrap our Bronze (Day 1) and Silver (Day 2) logic into Python functions. This is a core software engineering principle (DRY: Don't Repeat Yourself) that prepares our code for CI/CD.

**Databricks Workflows (Jobs)**: We will step out of the notebook and into the Databricks Jobs UI to configure a scheduled, multi-task workflow that orchestrates our modularized code automatically.

### Parameterizing the Notebook with Widgets
Widgets allow us to pass arguments to our notebook at runtime. We will use a dropdown widget to strictly control which layer of our Medallion Architecture executes.

In [0]:
# ---------------------------------------------------------
# WIDGET INITIALIZATION
# ---------------------------------------------------------
print("⚙️ Initializing Notebook Parameters...")

# Create a dropdown widget for the pipeline layer.
# Dropdowns act as input validation, preventing users from typing invalid layer names.
dbutils.widgets.dropdown(
    name="pipeline_layer", 
    defaultValue="silver", 
    choices=["bronze", "silver", "gold", "all"],
    label="Select Pipeline Layer:"
)

# Optional: Add a parameter for the target date (Useful for backfilling data)
dbutils.widgets.text("processing_date", "2019-11-01", "Processing Date (YYYY-MM-DD):")

# Retrieve the values passed by the user or the Job Orchestrator
current_layer = dbutils.widgets.get("pipeline_layer")
process_date = dbutils.widgets.get("processing_date")

print(f"✅ Configuration Loaded:")
print(f"   ➤ Target Layer: {current_layer.upper()}")
print(f"   ➤ Target Date:  {process_date}")

### Modularizing Pipeline Logic
Instead of executing code as a script, we wrap our data transformations inside functions. This makes testing easier and allows our "Controller" to route execution intelligently.

In [0]:
from pyspark.sql import functions as F
import time

# ---------------------------------------------------------
# MODULARIZED PIPELINE FUNCTIONS
# ---------------------------------------------------------

catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"

def run_bronze_layer():
    """Simulates the Day 1 logic: Ingesting CSV to Managed Delta."""
    print("🚀 [STARTED] Bronze Layer Ingestion...")
    start_time = time.time()
    
    # Normally, your actual Day 1 ingestion code goes here.
    # We will simulate a successful run for orchestration purposes.
    spark.sql(f"USE CATALOG {catalog_name}")
    table_name = f"{schema_name}.events_delta_managed"
    
    duration = round(time.time() - start_time, 2)
    print(f"✅ [COMPLETED] Bronze Layer. Validated table: {table_name} in {duration}s.")
    return "Success", duration

def run_silver_layer():
    """Executes the Day 2 logic: User-level feature aggregation."""
    print("🚀 [STARTED] Silver Layer Feature Engineering...")
    start_time = time.time()
    
    # Load Bronze
    df_bronze = spark.table(f"{catalog_name}.{schema_name}.events_delta_managed")
    
    # Perform a fast aggregation test to prove modularity works
    user_count = df_bronze.select("user_id").distinct().count()
    print(f"   ➤ Processed {user_count:,} distinct users.")
    
    duration = round(time.time() - start_time, 2)
    print(f"✅ [COMPLETED] Silver Layer in {duration}s.")
    return "Success", duration

def run_gold_layer():
    """Placeholder for future ML predictions/business aggregations."""
    print("🚀 [STARTED] Gold Layer Processing...")
    start_time = time.time()
    time.sleep(1) # Simulating work
    duration = round(time.time() - start_time, 2)
    print(f"✅ [COMPLETED] Gold Layer in {duration}s.")
    return "Success", duration

### The Execution Controller
This block reads the widget parameter and decides which function to trigger. This is exactly how production job tasks isolate their workloads.

In [0]:
# ---------------------------------------------------------
# EXECUTION ROUTER
# ---------------------------------------------------------
print(f"🚦 Routing execution for layer: {current_layer.upper()}...")

execution_log = []

try:
    if current_layer == "bronze" or current_layer == "all":
        status, duration = run_bronze_layer()
        execution_log.append(("Bronze", status, duration))
        
    if current_layer == "silver" or current_layer == "all":
        status, duration = run_silver_layer()
        execution_log.append(("Silver", status, duration))
        
    if current_layer == "gold" or current_layer == "all":
        status, duration = run_gold_layer()
        execution_log.append(("Gold", status, duration))

except Exception as e:
    print(f"❌ PIPELINE FAILED: {str(e)}")
    # In production, we explicitly raise the error to fail the Databricks Job
    raise e 

# ---------------------------------------------------------
# VISUALIZE EXECUTION REPORT
# ---------------------------------------------------------
# Creating a DataFrame to nicely display the run metrics
df_report = spark.createDataFrame(execution_log, ["Pipeline_Layer", "Status", "Duration_Seconds"])

print("\n📊 PIPELINE EXECUTION SUMMARY:")
display(df_report)

#### UI Instructions for Job Scheduling

### 👷‍♂️ UI Instructions: Creating the Production Job

Now that our notebook is parameterized, we need to schedule it using **Databricks Workflows**. Follow these steps in your Databricks Workspace:

1. **Navigate to Workflows:** Click on `Workflows` in the left-hand navigation sidebar, then click **Create Job**.
2. **Configure Task 1 (Bronze):**
   * **Task name:** `ingest_bronze`
   * **Type:** Notebook
   * **Source:** Select this current notebook.
   * **Parameters:** Click `+ Add`. Set Key = `pipeline_layer`, Value = `bronze`.
3. **Configure Task 2 (Silver):**
   * Click the `+` button to add a task.
   * **Task name:** `build_silver_features`
   * **Depends on:** `ingest_bronze` (This creates your DAG - Directed Acyclic Graph!).
   * **Source:** Select this *same* notebook.
   * **Parameters:** Key = `pipeline_layer`, Value = `silver`.
4. **Set the Schedule:**
   * On the right-hand panel of the Job screen, click **Add Schedule**.
   * Set it to run **Daily at 2:00 AM**.
5. **Run Now:** Click `Run Now` in the top right to test your orchestrated pipeline!